# Notebook 00 — Setup & Baseline Evaluation

**Inputs:** `data/test_prompts.json`, `data/gold_answers.json`  
**Outputs:** `results/baseline_base_model.json`  
**Runtime:** ~10 min on Kaggle T4

---
### Before running: fill gold answers
1. Open https://claude.ai — start a new chat with this system prompt:
> You are a DAA tutor. Guide the student through reasoning step by step rather than dumping the final answer. Ask leading questions, explain the intuition, build up the solution gradually. Keep each response to roughly 200-400 words.
2. Paste each prompt from `data/test_prompts.json` and copy the responses.
3. Save them into `data/gold_answers.json` (rename from `gold_answers_template.json`).

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys

PACKAGES = [
    "transformers>=4.45.0", "peft>=0.13.0", "trl>=0.11.0",
    "bitsandbytes>=0.44.0", "accelerate>=1.0.0", "datasets>=3.0.0",
    "sacrebleu", "bert-score", "huggingface_hub", "sentencepiece", "protobuf"
]
for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "transformers", "huggingface_hub"], check=False)
print("Done.")

## Cell 2 — Paths & HuggingFace login

In [ ]:
import os, sys, json
from pathlib import Path

KAGGLE = Path("/kaggle").exists()

if KAGGLE:
    PROJECT_ROOT = Path("/kaggle/working/daa-helper")
    if not PROJECT_ROOT.exists():
        import subprocess
        subprocess.run(["git", "clone",
                        "https://github.com/mimadraza/daa-helper.git",
                        str(PROJECT_ROOT)], check=False)
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))
print(f"Project root: {PROJECT_ROOT}")

# HF login
from huggingface_hub import login
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN") or input("Paste HF token: ").strip()
login(token=HF_TOKEN)
HF_USERNAME = "mimadraza"
print("Logged in.")

## Cell 3 — Load prompts and gold answers

In [ ]:
from utils.io_helpers import load_json, save_trial_result

prompts_data = load_json("test_prompts.json", base_dir="data")
gold_data    = load_json("gold_answers.json",  base_dir="data")

prompts      = [p["prompt"]      for p in prompts_data["prompts"]]
gold_answers = [a["gold_answer"] for a in gold_data["answers"]]

unfilled = [i+1 for i, g in enumerate(gold_answers)
            if "PASTE_CLAUDE" in g or not g.strip()]
if unfilled:
    raise ValueError(f"Gold answers not filled for prompts: {unfilled}")

print(f"Loaded {len(prompts)} prompts and {len(gold_answers)} gold answers.")
print(f"\nSample (prompt 1 gold answer):\n{gold_answers[0][:300]}...")

## Cell 4 — Load base model (4-bit QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "TinyLlama/TinyLlama_v1.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    use_safetensors=False,
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print(f"Loaded: {BASE_MODEL}")
print(f"Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Cell 5 — Run inference on 10 prompts

In [ ]:
from utils.evaluation import run_inference_on_prompts, evaluate_responses, free_memory

base_responses = run_inference_on_prompts(
    model, tokenizer, prompts,
    max_new_tokens=512, temperature=0.7
)

for i, (p, r) in enumerate(zip(prompts, base_responses)):
    print(f"\n--- Prompt {i+1} ---")
    print(f"Q: {p[:80]}...")
    print(f"A: {r[:200]}...")

## Cell 6 — Score against gold answers

In [ ]:
eval_results = evaluate_responses(base_responses, gold_answers)
agg = eval_results["aggregate"]

print("=" * 60)
print("BASE MODEL EVALUATION")
print("=" * 60)
print(f"Mean BLEU:         {agg['mean_bleu']:.2f}")
print(f"Mean BERTScore F1: {agg['mean_bertscore_f1']:.4f}")
print(f"Combined score:    {agg['combined_score']:.4f}")
print("\nPer-prompt:")
for i, pp in enumerate(eval_results["per_prompt"]):
    print(f"  Prompt {i+1}: BLEU={pp['bleu']:.2f}, BERTScore F1={pp['bertscore_f1']:.4f}")

## Cell 7 — Save results

In [ ]:
sample_responses = [
    {"prompt": p, "response": r, "gold": g}
    for p, r, g in zip(prompts, base_responses, gold_answers)
]
save_trial_result(
    trial_name="base_model", stage="baseline",
    config={"model": BASE_MODEL, "quantization": "4-bit nf4"},
    eval_results=eval_results, train_metrics={},
    sample_responses=sample_responses,
)
print("Saved → results/baseline_base_model.json")

del model
free_memory()
print("\n✓ Notebook 00 complete. Run Notebook 02 next.")

## Cell 8 — (Optional) Push results to HF Hub

In [ ]:
# Uncomment to back up results to a HF dataset repo
# from utils.io_helpers import push_to_hf_hub
# push_to_hf_hub(
#     local_dir=str(PROJECT_ROOT / "results"),
#     repo_id=f"{HF_USERNAME}/daa-helper-results",
#     repo_type="dataset",
#     commit_message="baseline results",
# )